In [1]:
from pyod.models.kde import KDE
from pyod.utils.example import visualize
from sklearn.preprocessing import MinMaxScaler

import pandas as pd
import numpy as np

import os
import pickle

from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt 
%matplotlib inline

import sys
sys.path.append('..')
sys.path.append('../..')

from src.utils import train_test_anomaly, raw_thresholds

In [2]:
file_path = '../../datasets/Dodgers/101-freeway-traffic.test.out'

columns = ['value', 'anomaly']

df = pd.read_csv(file_path, names=columns, header=None)

In [3]:
df = df[df['value'] >= 0]
df['anomaly'].value_counts()

anomaly
0    44788
1     2709
Name: count, dtype: int64

In [4]:
train_data, test_data = train_test_anomaly(data=df, shuffle=False)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(df, test_size=0.3, shuffle=False)
print(X_train.shape)
print(X_test.shape)

(33247, 2)
(14250, 2)


In [6]:
print(X_train['anomaly'].value_counts())
print(X_test['anomaly'].value_counts())

anomaly
0    31437
1     1810
Name: count, dtype: int64
anomaly
0    13351
1      899
Name: count, dtype: int64


In [7]:
train_np = X_train[['value']][X_train['anomaly'] == 0]
train_np

,value
379,23
380,42
381,37
382,24
383,39
...,...
35361,24
35362,37
35363,36
35364,39


In [8]:
model = KDE(contamination=0.057)
model.fit(train_np)

KDE(algorithm='auto', bandwidth=1.0, contamination=0.057, leaf_size=30,
  metric='minkowski', metric_params=None)

In [13]:
anomaly_scores = model.decision_function(X_test[['value']])
print(f'max score: {np.max(anomaly_scores)}')
print(f'min score: {np.min(anomaly_scores)}')

max score: 10.515739775823658
min score: 3.2628130655167933


In [14]:
thres = raw_thresholds(anomaly_scores, contamination=0.062)
thres

4.5034531497478465

In [15]:
thres_np = [1.0 if (score > thres - 0.25) else 0 for score in anomaly_scores]
len(thres_np)

14250

In [16]:
gtruth_np = X_test[['anomaly']]
gtruth_np

,anomaly
35391,1
35392,1
35393,1
35394,1
35395,1
...,...
50109,0
50110,0
50111,0
50112,0


In [17]:
prec = precision_score(gtruth_np, thres_np, pos_label=1)
recall = recall_score(gtruth_np, thres_np, pos_label=1)
f1 = f1_score(gtruth_np, thres_np, pos_label=1)

In [18]:
print(f'Precsion Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precsion Score: 0.0390  Recall: 0.0512  f1_score: 0.0443


In [22]:
visualize('KDE', X_train=X_train['value'], y_train=X_train[['anomaly']], y_train_pred=X_train[['anomaly']], X_test=X_test[['value']], y_test=X_test[['anomaly']], y_test_pred=thres_np, show_figure=True)

ValueError: Expected 2D array, got 1D array instead:
array=[23 42 37 ... 22 20 19].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.